In [1]:
from pathlib import Path
import pandas as pd

PROJECT_DIR = Path("..")

RAW_DIR = PROJECT_DIR / "data" / "Raw"

DK_PATH = (
    PROJECT_DIR
    / "data"
    / "temp_data"
    / "parliament"
    / "roll_calls_translated.csv"
)

RAW_FILES = [
    "P01_05_DK.csv",
    "P05_07_DK.csv",
    "P07_11_DK.csv",
    "P11_15_DK.csv",
    "P15_19_DK.csv",
    "P19_22_DK.csv",
    "P22_present_DK.csv"
]

In [2]:
raw_parts = []

for filename in RAW_FILES:

    path = RAW_DIR / filename

    df = pd.read_csv(
        path,
        usecols=["afstemningid", "dato"]
    )

    df["source_period"] = filename

    raw_parts.append(df)

raw_votes = pd.concat(
    raw_parts,
    ignore_index=True
)

print("Total raw voting records:", len(raw_votes))

display(raw_votes.head())

Total raw voting records: 1026339


,afstemningid,dato,source_period
0,5973,2004-10-19 13:15:00,P01_05_DK.csv
1,5973,2004-10-19 13:15:00,P01_05_DK.csv
2,5973,2004-10-19 13:15:00,P01_05_DK.csv
3,5973,2004-10-19 13:15:00,P01_05_DK.csv
4,5973,2004-10-19 13:15:00,P01_05_DK.csv


In [4]:
raw_votes["dato"] = pd.to_datetime(
    raw_votes["dato"],
    errors="coerce"
)

raw_votes["dk_year"] = raw_votes["dato"].dt.year

# Check whether any vote IDs have conflicting dates
date_conflicts = (
    raw_votes.groupby("afstemningid")["dato"]
    .nunique()
)

print(
    "Vote IDs with multiple dates:",
    (date_conflicts > 1).sum()
)

# One record per unique vote
vote_years = (
    raw_votes
    .dropna(subset=["dk_year"])
    .sort_values("dato")
    .drop_duplicates(subset=["afstemningid"])
    [["afstemningid", "dk_year"]]
    .copy()
)

vote_years["dk_year"] = vote_years["dk_year"].astype(int)

print("Unique Danish votes:", len(vote_years))

display(vote_years.head())

Vote IDs with multiple dates: 0
Unique Danish votes: 9870


,afstemningid,dk_year
1148,5986,2004
118,5973,2004
25200,6227,2004
24109,6216,2004
28463,6259,2004


In [5]:
# ============================================================
# MATCH ACTUAL VOTING YEAR TO TRANSLATED TITLES
# ============================================================

df_translated = pd.read_csv(DK_PATH)

# Avoid duplicate columns when rerunning
df_translated = df_translated.drop(
    columns=["dk_year"],
    errors="ignore"
)

df_translated = df_translated.merge(
    vote_years,
    on="afstemningid",
    how="left",
    validate="many_to_one"
)

print("Total translated votes:", len(df_translated))

print(
    "Votes with matched year:",
    df_translated["dk_year"].notna().sum()
)

print(
    "Votes without matched year:",
    df_translated["dk_year"].isna().sum()
)

display(
    df_translated[
        [
            "afstemningid",
            "sag_titel_en",
            "dk_year"
        ]
    ].head(10)
)

Total translated votes: 2128
Votes with matched year: 1873
Votes without matched year: 255


,afstemningid,sag_titel_en,dk_year
0,2,Proposal for a law amending the Corporate Tax ...,2014.0
1,1783,Proposal for a law amending the Law on the tax...,2014.0
2,2267,Proposal for a law amending the Law on Product...,2014.0
3,2268,Draft law on Danish tourism.,2014.0
4,2282,Proposal for a law amending the Act on Active ...,2014.0
5,2283,Proposal for a law amending the law on holiday...,2014.0
6,2284,Proposal for a law amending the law on the sta...,2014.0
7,2285,Proposal for a law repealing the law on nutrit...,2014.0
8,2286,Proposal for a law amending the Competition Ac...,2014.0
9,2287,Draft law on the Danish Green Investment Fund.,2014.0


In [7]:
unmatched = df_translated[
    df_translated["dk_year"].isna()
]

print("Unmatched votes:", len(unmatched))

# Keep unmatched years as missing values
df_translated["dk_year"] = df_translated["dk_year"].astype("Int64")

# Backup original file before overwriting
backup_path = DK_PATH.with_name("roll_calls_translated_backup.csv")

if not backup_path.exists():
    import shutil
    shutil.copy2(DK_PATH, backup_path)

# Save updated CSV
df_translated.to_csv(
    DK_PATH,
    index=False,
    encoding="utf-8-sig"
)

print("File saved:", DK_PATH)
print("Total rows:", len(df_translated))
print("Rows with year:", df_translated["dk_year"].notna().sum())
print("Rows without year:", df_translated["dk_year"].isna().sum())

Unmatched votes: 255
File saved: ..\data\temp_data\parliament\roll_calls_translated.csv
Total rows: 2128
Rows with year: 1873
Rows without year: 255
